# QLoRA Fine-Tuning: Legal Demand Assistant

Fine-tunes **Qwen3.5-9B** on a 50-example legal instruction dataset using QLoRA (4-bit quantization + LoRA adapters).

**Requirements:** Colab Pro with A100 GPU (~15-20 GB VRAM)

| Parameter | Value |
|---|---|
| LoRA rank | 16 |
| LoRA alpha | 32 |
| Epochs | 3 |
| Learning rate | 2e-4 |
| Effective batch size | 8 (2 × 4 grad accum) |
| Max sequence length | 1024 |

## Section 0: Install Dependencies

In [ ]:
!pip install transformers>=4.46.0 peft>=0.13.0 bitsandbytes>=0.44.0 trl>=0.12.0 datasets accelerate scikit-learn

In [ ]:
# This compiles from source and takes 5-10 min — watch the output
!pip install flash-attn --no-build-isolation

## Section 1: Imports and Configuration

In [ ]:
import json
import os

import torch
from datasets import Dataset
from peft import LoraConfig, PeftModel
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from trl import SFTConfig, SFTTrainer

# ── Hyperparameters ──────────────────────────────────────────────────────────

MODEL_ID = "Qwen/Qwen3.5-9B"
OUTPUT_DIR = "./qlora-legal-demand-adapter"
DATA_PATH = "./data/instruction_dataset.json"  # Update if your repo is cloned elsewhere

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
LR_SCHEDULER = "cosine"
PER_DEVICE_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
MAX_SEQ_LENGTH = 1024
WEIGHT_DECAY = 0.01

SYSTEM_PROMPT = (
    "You are a legal demand assistant. You help draft demand letters, "
    "identify legal claims, extract elements from demand letters, "
    "evaluate letter quality, and recommend remedies."
)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## Section 2: Upload / Clone Dataset

Upload `instruction_dataset.json` or clone the repo. Adjust `DATA_PATH` above if needed.

In [ ]:
# Option A: Clone the repo (uncomment and update URL)
# !git clone https://github.com/BigDataAnalytics-CS5542/Lab_8.git
# DATA_PATH = "./Lab_8/data/instruction_dataset.json"

# Option B: Upload the file via Colab UI
# from google.colab import files
# uploaded = files.upload()  # upload instruction_dataset.json
# DATA_PATH = "./instruction_dataset.json"

# Verify the file exists
assert os.path.exists(DATA_PATH), f"Dataset not found at {DATA_PATH} — update DATA_PATH"
print(f"Dataset found: {DATA_PATH}")

## Section 3: Load and Format Dataset

In [ ]:
with open(DATA_PATH) as f:
    raw_data = json.load(f)

print(f"Loaded {len(raw_data)} examples")
print(f"Task types: {set(ex['task_type'] for ex in raw_data)}")

# Format into ChatML messages
formatted = []
for example in raw_data:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{example['instruction']}\n\n{example['input']}"},
        {"role": "assistant", "content": example["output"]},
    ]
    formatted.append({
        "messages": messages,
        "task_type": example["task_type"],
    })

dataset = Dataset.from_list(formatted)

# Stratified 90/10 split (45 train / 5 val) by task_type
task_types = dataset["task_type"]
indices = list(range(len(dataset)))

train_idx, val_idx = train_test_split(
    indices, test_size=5, random_state=42, stratify=task_types
)

train_ds = dataset.select(train_idx).remove_columns(["task_type"])
val_ds = dataset.select(val_idx).remove_columns(["task_type"])

print(f"Train: {len(train_ds)} examples, Val: {len(val_ds)} examples")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Use flash_attention_2 if available, otherwise fall back to eager
try:
    import flash_attn
    attn_impl = "flash_attention_2"
    print("Using flash_attention_2")
except ImportError:
    attn_impl = "eager"
    print("flash-attn not installed — using eager attention (still works, just slower)")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    attn_implementation=attn_impl,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False

print(f"Model loaded: {MODEL_ID}")
print(f"Model memory: {model.get_memory_footprint() / 1e9:.2f} GB")

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    attn_implementation="flash_attention_2",
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False

print(f"Model loaded: {MODEL_ID}")
print(f"Model memory: {model.get_memory_footprint() / 1e9:.2f} GB")

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    task_type="CAUSAL_LM",
    bias="none",
)

training_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER,
    weight_decay=WEIGHT_DECAY,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=5,
    report_to="none",
    seed=42,
)

print("LoRA config:", peft_config)
print(f"\nTraining: {NUM_EPOCHS} epochs, lr={LEARNING_RATE}, effective batch={PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")

In [ ]:
peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    task_type="CAUSAL_LM",
    bias="none",
)

training_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER,
    weight_decay=WEIGHT_DECAY,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_seq_length=MAX_SEQ_LENGTH,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=5,
    report_to="none",
    seed=42,
)

print("LoRA config:", peft_config)
print(f"\nTraining: {NUM_EPOCHS} epochs, lr={LEARNING_RATE}, effective batch={PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")

trainer = SFTTrainer(
    model=model,
    args=training_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=peft_config,
    processing_class=tokenizer,
    max_seq_length=MAX_SEQ_LENGTH,
)

print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print("Starting training...")

result = trainer.train()

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=peft_config,
    processing_class=tokenizer,
)

print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print("Starting training...")

result = trainer.train()

In [ ]:
# Print training results
print("── Training Results ──")
print(f"  Total steps:    {result.global_step}")
print(f"  Training loss:  {result.training_loss:.4f}")
for key, value in result.metrics.items():
    print(f"  {key}: {value}")

## Section 7: Save Adapter Weights

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to {OUTPUT_DIR}")

# List saved files
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f:40s} {size / 1e6:.1f} MB")

# Free training memory before inference
del trainer, model
torch.cuda.empty_cache()

# Reload base model + adapter
bnb_config_inf = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config_inf,
    attn_implementation=attn_impl,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
model.eval()
print("Model + adapter loaded for inference")

In [ ]:
# Free training memory before inference
del trainer, model
torch.cuda.empty_cache()

# Reload base model + adapter
bnb_config_inf = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config_inf,
    attn_implementation="flash_attention_2",
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
model.eval()
print("Model + adapter loaded for inference")

In [ ]:
test_prompts = [
    {
        "task": "Demand Letter",
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": (
                "Draft a legal demand letter based on the scenario.\n\n"
                "A dog walker lost a client's dog due to negligence. The dog was "
                "found two days later, but the owner incurred $1,200 in search and "
                "veterinary costs."
            )},
        ],
    },
    {
        "task": "Claim Identification",
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": (
                "Identify the strongest legal claim based on the scenario.\n\n"
                "A mechanic charged $800 for car repairs that were never actually performed."
            )},
        ],
    },
    {
        "task": "Element Extraction",
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": (
                "Extract the key legal elements from this demand letter.\n\n"
                "Dear QuickFix Plumbing, I represent Maria Santos regarding "
                "water damage totaling $3,600 caused by your faulty pipe installation "
                "on February 15, 2026. Please remit payment within 14 days."
            )},
        ],
    },
]

for prompt in test_prompts:
    text = tokenizer.apply_chat_template(
        prompt["messages"],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )

    print(f"\n{'='*60}")
    print(f"Task: {prompt['task']}")
    print(f"{'='*60}")
    print(response.strip())

## Section 9: Download Adapter (Optional)

Download the adapter weights to your local machine for use in the FastAPI pipeline.

In [ ]:
# Zip and download the adapter
!zip -r qlora-legal-demand-adapter.zip {OUTPUT_DIR}

# Uncomment to download via Colab UI
# from google.colab import files
# files.download("qlora-legal-demand-adapter.zip")